# Copper LSTM?SA research (Kaggle)

Read `audit_report.md` and `README_KAGGLE.md`. This notebook runs one or more phases through `research.py`. `QUICK_TEST` uses a short recent data slice and never counts as a research result. `FULL_RESEARCH` uses the complete saved data snapshot. Keep Internet and GPU T4 enabled for phase 1. Upload this notebook together with `research.py` and `experiment_config.json` as a Kaggle Dataset.

In [ ]:
from pathlib import Path
import shutil, sys, subprocess, importlib.util

PHASES = ["phase1"]  # choose from phase1, phase2, phase3, phase4, phase5
MODE = "QUICK_TEST"   # change to FULL_RESEARCH after checking the quick run
ALLOW_LARGE = False   # set True after reading compute_estimate.json

for package in ["yfinance"] + (["optuna"] if "phase3" in PHASES else []):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

source_dirs = [Path.cwd()] + [p.parent for p in Path("/kaggle/input").rglob("research.py")]
source = next((p for p in source_dirs if (p / "research.py").exists() and (p / "experiment_config.json").exists()), None)
if source is None:
    raise FileNotFoundError("Attach a Kaggle Dataset containing research.py and experiment_config.json")
local = Path("/kaggle/working/copper_research_src")
local.mkdir(parents=True, exist_ok=True)
for name in ["research.py", "experiment_config.json"]:
    shutil.copy2(source / name, local / name)
sys.path.insert(0, str(local))
print("Source:", source, "| working copy:", local)

In [ ]:
import research
for phase in PHASES:
    print("\n===", MODE, phase, "===")
    research.run(phase, mode=MODE, allow_large=ALLOW_LARGE)
print("Artifacts:", research.output_dir(research.load_config(MODE)))

## Phase guide

1. Run `QUICK_TEST` through phases 1?5 to validate plumbing.
2. Change `MODE` to `FULL_RESEARCH` and run phase 1. Inspect its benchmark and data quality report.
3. Run phase 2; review screening results and the selection rule.
4. Run phase 3. It prints a projection from the measured pilot. If it exceeds four hours, inspect `compute_estimate.json` and set `ALLOW_LARGE=True` to start the search. The SA JSON state and Optuna SQLite study resume on the next Kaggle session.
5. Run phase 4 for five training seeds, search seeds, split ratios, and expanding folds.
6. Run phase 5 once selection is frozen. It is the only phase that creates test windows. Download the output folder including `summary_report.md` and all CSV/PNG artifacts.

A Kaggle session can end during any phase. Rerun the same phase with the same mode and files to continue. Keep the output folder in a Kaggle Dataset between fresh sessions; otherwise the local `/kaggle/working` state is lost.